In [1]:
import json
import os, sys
import pickle
import datetime
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn import preprocessing
from sklearn import metrics
import seaborn as sns
import ROOT as rt


## user define
sys.path.append(".")
sys.path.append("./Utils")
from Plotter import HistStack
from CommonTrain import *
from Model import Seq

2023-04-30 20:10:10.965885: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /cvmfs/sft.cern.ch/lcg/releases/MCGenerators/thepeg/2.2.3-6cb32/x86_64-centos7-gcc11-opt/lib/ThePEG:/cvmfs/sft.cern.ch/lcg/releases/MCGenerators/herwig++/7.2.3-90caf/x86_64-centos7-gcc11-opt/lib/Herwig:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/jaxlib/mlir/_mlir_libs:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/torch/lib:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/onnxruntime/capi/:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorflow:/cvmfs/sft.cern.ch/lcg/views/LCG_102b_swan/x86_64-centos7-gcc11-opt/lib/python3.9/site-packages/tensorfl

Welcome to JupyROOT 6.26/08


# Set config

In [2]:
setting={}
with open('config/Settrain.json') as json_file:
    data = json.load(json_file)
    setting=data
INPUTDIR = setting["INPUTDIR"]
VERSION = setting["VERSION"]
# SIGNAL_MASSES = ["Zp-1700_CH-345", "Zp-2900_CH-345", "Zp-4100_CH-345", "Zp-2900_CH-1095", "Zp-4100_CH-1095", "Zp-4100_CH-1845"]
CHANNEL="ee"
# ERA="ee17"
print(VERSION)
INPUTDIR = sys.argv[1]
REGION = sys.argv[2]

# for dev
# INPUTDIR = setting["INPUTDIR"]+'/ee18/'
# INPUTDIR = '/eos/user/h/hkwon/Run2/Preselection/20230430wp80/Presel/ee17/'
# REGION = 'SR'

2803_equalpropv1


# Load data

In [3]:
if REGION == 'SR':
    file = 'nested_dic_df.pkl'
if REGION == 'VR':   
    file = 'nested_dic_df_PT.pkl'
# file = "/"+ERA+"/nested_dic_df.pkl"
with open(INPUTDIR+'/'+file, 'rb') as f:
    nested_dic_df = pickle.load(f)
 
dic_df=nested_dic_df['raw']
dic_df_met_u_unsort=nested_dic_df['met_u']
dic_df_met_d_unsort=nested_dic_df['met_d'] 

# # temp!!
for key in dic_df_met_u_unsort:
    df = dic_df_met_u_unsort[key]
#     print(df[df.isin([np.inf, -np.inf]).any(1)])
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_u_unsort[key] = df.drop(i)
    df = dic_df_met_d_unsort[key]
    if df.isnull().values.any():
        print("Warning, nan in",key)
        i = df[df.isin([np.nan, np.inf, -np.inf]).any(1)].index
        print(i)
        dic_df_met_d_unsort[key] = df.drop(i)
        
selected_columns = [
#                     'MC1C1_pre', 'H3PP_pre', 'H3TPP_pre', 
                    'pt_lead_pre', 'mass_pre', # by auc rank
                    'MT_pre', 'pt_trail_pre', 'Zpt_pre', 'dRll_pre', 'PuppiMET_pre',
                    'dPhilMET_pre', 'dPhillMET_pre', 'U_pre', 'dPhill_pre', 'MT2_pre',
                    'JZB_pre'] # order used in training

# # dic_df_meta={}
# # check b jet
# query_str="nbjet_pre==0"

# print("####################################################")
# print("####   (unweighted) #events after preselection  ####")
# print("####################################################")
# for key in dic_df:  
# #     dic_df_meta[key]=dic_df[key].query(query_str)
#     dic_df[key]=dic_df[key].query(query_str) #temp!
#     print(key, ":", dic_df[key].shape[0])
    
dic_df_train={}    
dic_df_met_u={}
dic_df_met_d={}
for key in dic_df:
    print(key)
    #dic_df_train[key]=pd.concat([dic_df[key][selected_columns], dic_df1[key][selected_columns]]) #when use ll channel
    #dic_df[key]=pd.concat([dic_df[key], dic_df1[key]])
#     dic_df_train[key]=dic_df_meta[key][selected_columns] # check b jet   
    dic_df_train[key]=dic_df[key][selected_columns]    
    dic_df_met_u[key]=dic_df_met_u_unsort[key][selected_columns]    
    dic_df_met_d[key]=dic_df_met_d_unsort[key][selected_columns]

Zp-4100_CH-1345
DY1500To2000
Zp-1700_CH-345
Zp-2500_CH-345
DY500To700
Zp-1700_CH-845
Zp-4100_CH-595
Zp-3300_CH-345
Zp-1700_CH-595
Zp-3700_CH-345
Zp-2100_CH-595
Zp-2900_CH-345
Zp-4100_CH-1095
Zp-2900_CH-845
Zp-3700_CH-1345
Zp-3300_CH-1595
Zp-2500_CH-595
Zp-4100_CH-345
Zp-3700_CH-1845
Zp-2500_CH-845
Zp-4100_CH-1595
Zp-4100_CH-1845
Zp-3300_CH-595
Zp-2100_CH-345
Zp-2900_CH-595
Zp-4100_CH-845
Zp-2100_CH-845
Zp-3300_CH-845
Zp-3300_CH-1345
Zp-3700_CH-1595
DY2000To3000
Zp-3700_CH-595
DY700To800
Zp-2900_CH-1345
Zp-3700_CH-1095
DY800To1000
DY1000To1500
Zp-3700_CH-845
Zp-3300_CH-1095
Zp-2900_CH-1095
TT
TThdampUp
TThdampDown
TTUp
TTDown
TTSemi
TTZ
DY
top
antitop
ZZ
WZ
WW
Data


In [4]:
print(dic_df["Zp-2500_CH-345"].columns)
print(dic_df_train["Zp-2500_CH-345"].columns)
print(dic_df_met_u["Zp-2500_CH-345"].columns)

Index(['PuppiMET_pre', 'mass_pre', 'pt_lead_pre', 'pt_trail_pre', 'U_pre',
       'JZB_pre', 'dPhill_pre', 'dPhilMET_pre', 'MT_pre', 'MT2_pre', 'Zpt_pre',
       'dRll_pre', 'dPhillMET_pre', 'MET_pre', 'MET_phi_pre',
       'PuppiMET_phi_pre', 'eta_lead_pre', 'eta_trail_pre', 'phi_lead_pre',
       'phi_trail_pre', 'njet_pre', 'nbjetflav_pre', 'genweight_u_pre',
       'genweight_d_pre', 'genweight_pre', 'puweight_pre', 'puweight_u_pre',
       'puweight_d_pre', 'l1prefiring_pre', 'l1prefiring_u_pre',
       'l1prefiring_d_pre', 'effSF_pre', 'effSF_u_pre', 'effSF_d_pre',
       'trigSF_pre', 'trigSF_u_pre', 'trigSF_d_pre', 'topSF_pre',
       'topSF_u_pre', 'topSF_d_pre', 'btagSF_pre', 'weight', 'wopu', 'wol1',
       'woeff', 'wotop', 'wobsf', 'weighthdampUp', 'weighthdampDown',
       'weightPSUp', 'weightPSDown', 'weightpuweightUp', 'weightpuweightDown',
       'weightl1prefiringUp', 'weightl1prefiringDown', 'weighteffSFUp',
       'weighteffSFDown', 'weighttopSFUp', 'weighttopSFDow

In [5]:
def get_x(sig_key, dic_df_train):
    # x: sig and all bkgs concatenated
    x = dic_df_train[sig_key].copy(deep=True)
    for k in dic_df_train:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_df_train[k]], axis=0)
    
    return x

In [6]:
def get_xy(sig_key, dic_predict_sort):
    # x: sig and all bkgs concatenated
    x = dic_predict_sort[sig_key].copy(deep=True)
    for k in dic_predict_sort:
        # filter data out
        if "Data" in k:
            continue
        if "Zp" not in k:
            x = np.concatenate([x, dic_predict_sort[k]], axis=0)
            
    tot_sig = dic_predict_sort[sig_key].shape[0]
    tot_bkgs = x.shape[0] - tot_sig        
    y_sig = np.full((tot_sig,1), 1)
    y_bkgs = np.full((tot_bkgs,1), 0) 
    y = np.concatenate((y_sig, y_bkgs), axis=0) 
    
    return x, y

In [18]:
import math
sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "met", "trigSF", "hdamp", "PS"]

# sys_list = ["l1prefiring"]
if REGION == 'SR':
    threshold = 0.3
else:
    threshold = 1    
    
def draw_with_systematics(model_mass, dic_df_raw, dic_nested_shape_sys, sys_list, variable, r, region):
    dic_sys = {}
    dic_sys_sqr = {}
    # nominal weight plot
    file_prefix = "Outputs/"+REGION
    hist=HistStack(dic_df_raw, variable, 'weight', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/norm_dnnscore.pdf", blind_from=threshold)     
    dic_yields = hist.get_yields_by_bin()
    bkg_norm = dic_yields['bkg']
    hist.create_root_file(file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    
    for s in sys_list:
        if "met" in s or "scale" in s: # fill later
            continue
        # up
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Up', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systUp_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysUp = dic_yields['bkg']
        dic_sys[s+'Up'] = np.array(sysUp) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        # down
        hist=HistStack(dic_df_raw, variable, 'weight'+s+'Down', r[0], r[1], r[2], r[3])            
        c1=hist.draw_hist(model_mass, ERA, file_name=file_prefix+"/sys/systDown_"+s+".pdf", blind_from=threshold)     
        dic_yields = hist.get_yields_by_bin()
        sysDown = dic_yields['bkg']
        dic_sys[s+'Down'] = np.array(sysDown) - np.array(bkg_norm)
        hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        
    # shape sys
    hist=HistStack(dic_nested_shape_sys['met_u'], variable, 'weightmetUp', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist("Zp-2500_CH-345", ERA, file_name=file_prefix+"/sys/systUp_"+s+".pdf")     
    dic_yields = hist.get_yields_by_bin()
    sysUp = dic_yields['bkg']
    dic_sys['metUp'] = np.array(sysUp) - np.array(bkg_norm)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
    hist=HistStack(dic_nested_shape_sys['met_d'], variable, 'weightmetDown', r[0], r[1], r[2], r[3])            
    c1=hist.draw_hist("Zp-2500_CH-345", ERA, file_name=file_prefix+"/sys/systDown_"+s+".pdf")     
    dic_yields = hist.get_yields_by_bin()
    sysUp = dic_yields['bkg']
    dic_sys['metDown'] = np.array(sysUp) - np.array(bkg_norm)
    hist.export_hist(model_mass, file_name=file_prefix+"/shape_v1_"+VERSION+model_mass)
        
    # si^2
    for s in sys_list:
        # by source
        sqrUp = [x*x if x>0 else -x*x for x in dic_sys[s+'Up']]
        sqrUp_up = [x if x>0 else 0 for x in sqrUp]
        sqrUp_down = [x if x<0 else 0 for x in sqrUp]
        sqrDown = [x*x if x>0 else -x*x for x in dic_sys[s+'Down']]
        sqrDown_up = [x if x>0 else 0 for x in sqrDown]
        sqrDown_down = [x if x<0 else 0 for x in sqrDown]
        
        sqr_up = np.array(sqrUp_up) + np.array(sqrDown_up)
        sqr_down = np.array(sqrUp_down) + np.array(sqrDown_down)
        
        dic_sys_sqr[s+'Up'] = sqr_up
        dic_sys_sqr[s+'Down'] = np.fabs(sqr_down) #store absolute
        
    print(dic_sys)
    print(dic_sys_sqr)
    # sqrt(sumi si^2)
    for i, s in enumerate(sys_list):
        if i==0:
            sum_up = dic_sys_sqr[s+'Up'].copy() # be carefull not to use shallow copy
            sum_down = dic_sys_sqr[s+'Down'].copy()
        else:
            sum_up += dic_sys_sqr[s+'Up']
            sum_down += dic_sys_sqr[s+'Down']

    dic_sys_sqr['sum_up'] = np.sqrt(sum_up)
#     print(sum_up)
#     print(sum_down)
    dic_sys_sqr['sum_down'] = np.sqrt(sum_down)
    print(dic_sys_sqr)
            
    return dic_sys_sqr

In [19]:
from multiprocessing import Pool
def wrapper_mp(model_mass):
    
#     # dump roc
#     dic_roc = {}
#     x, y = get_xy(model_mass, dic_predict_sort)
#     tpr, fpr, auc = dump_roc(model_mass, y, x, dic_predict_sort)
#     dic_roc['tpr'] = tpr
#     dic_roc['fpr'] = fpr
#     dic_roc['auc'] = auc
#     dic_nested_roc[model_mass] = dic_roc            


    #-- implement systematics --#   
    os.system("mkdir -p Outputs/"+REGION)
    os.system("mkdir -p Outputs/"+REGION+"/sys")
#     sys_list = ["puweight", "l1prefiring", "effSF", "topSF", "trigSF"]
    # sys_list = ["puweight"]

    r = [nbin, 0, 1, "DNN score"]

    if REGION == 'SR':
        threshold = 0.3
    else:
        threshold = 1    
    
    # draw plot with syst
    dic_sys_sqr = draw_with_systematics(model_mass, dic_predict_sort, dic_nested_shape_sys, sys_list, "score"+model_mass, r, "region_dummy")

    hist=HistStack(dic_predict_sort, "score"+model_mass, 'weight', nbin, 0, 1, "DNN score", dic_sys=dic_sys_sqr) 
    
    c1=hist.draw_hist(model_mass, ERA, file_name="Outputs/"+REGION+"/"+model_mass, blind_from=threshold)     
    hist.get_yeilds()
    hist.draw_pie(file_name="Outputs/"+REGION+"/"+model_mass)     
    hist.export_yields_table(file_name="Outputs/"+REGION+"/"+model_mass)     
     
    return None

# Load model and evaluate

In [20]:
import time

nbin=20
# VERSION = '0903'
# VERSION = setting["VERSION"]
print(time.time())

one_mass = 'Zp-2500_CH-345'

if 'mm' in INPUTDIR:
    CHANNEL = 'mm'
if 'ee' in INPUTDIR:
    CHANNEL = 'ee'
if 'em' in INPUTDIR:
    CHANNEL = 'em'
    
OUTDIR = 'Outputs/'+CHANNEL
file_postfix = 'PNN_'+VERSION
# file_postfix = one_mass+VERSION

# model_masses = ['Zp-1700_CH-345', 'Zp-2500_CH-345', 'Zp-2900_CH-345', 'Zp-3300_CH-345', 'Zp-4100_CH-345']
model_masses = [m for m in dic_df_train if 'Zp' in m] # if want to run all
# model_masses = ['Zp-2500_CH-845']

# %matplotlib inline
rt.gROOT.SetBatch(rt.kTRUE)

dic_nested_roc = {}

ERA = INPUTDIR

model = keras.models.load_model(OUTDIR+'/model_'+file_postfix)
#     model = keras.models.load_model(OUTDIR+'/model_binary_'+file_postfix)

Transformer = pickle.load(open(OUTDIR+'/scaler_'+file_postfix+'.pkl','rb'))

nested_dic_predict, nested_dic_predict_met_u, nested_dic_predict_met_d = [{} for _ in range(3)]

# masses loop
for model_mass in model_masses:
    masses = model_mass.split('_')
    zp_m = masses[0].split('-')
    zp_m = zp_m[1]
    ch_m = masses[1].split('-')
    ch_m = ch_m[1]
    
#     file_postfix = 'test'

    dic_df_scale, dic_df_scale_met_u, dic_df_scale_met_d = [{} for _ in range(3)]        

    for key2 in dic_df_train:
        dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
        dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
        dic_df_met_u[key2]['zp_m'] = np.full(dic_df_met_u[key2].shape[0], int(zp_m))
        dic_df_met_u[key2]['ch_m'] = np.full(dic_df_met_u[key2].shape[0], int(ch_m))
        dic_df_met_d[key2]['zp_m'] = np.full(dic_df_met_d[key2].shape[0], int(zp_m))
        dic_df_met_d[key2]['ch_m'] = np.full(dic_df_met_d[key2].shape[0], int(ch_m))
        if dic_df_train[key2].empty:
            continue
        dic_df_scale[key2] = Transformer.transform(dic_df_train[key2]) 
        dic_df_scale_met_u[key2] = Transformer.transform(dic_df_met_u[key2]) 
        dic_df_scale_met_d[key2] = Transformer.transform(dic_df_met_d[key2]) 

    # predict
    dic_predict, dic_predict_met_u, dic_predict_met_d = [{} for _ in range(3)]

    for key3 in dic_df_scale:
    #             if "Zp-" in key3 and not key in key3: # use same scaling used in model training for all masses
    #                 continue
        dic_predict[key3] = model.predict(dic_df_scale[key3], batch_size=dic_df_scale[key3].shape[0])
        dic_predict_met_u[key3] = model.predict(dic_df_scale_met_u[key3], batch_size=dic_df_scale_met_u[key3].shape[0])
        dic_predict_met_d[key3] = model.predict(dic_df_scale_met_d[key3], batch_size=dic_df_scale_met_d[key3].shape[0])
    
    nested_dic_predict[model_mass] = dic_predict
    nested_dic_predict_met_u[model_mass] = dic_predict_met_u
    nested_dic_predict_met_d[model_mass] = dic_predict_met_d
    

# array in y-axis to dataframe
dic_predict_sort, dic_predict_met_u_sort, dic_predict_met_d_sort = [{} for _ in range(3)]

for key4 in dic_df_scale:
    new_df = dic_df[key4].copy(deep=True).reset_index(drop="True")
    new_df_met_u = pd.DataFrame()
    new_df_met_d = pd.DataFrame()
    new_df_met_u["weightmetUp"] = dic_df_met_u_unsort[key4]["weightmetUp"].copy(deep=True).reset_index(drop="True")
    new_df_met_d["weightmetDown"] = dic_df_met_d_unsort[key4]["weightmetDown"].copy(deep=True).reset_index(drop="True")    
    dic_predict_sort[key4] = new_df
    dic_predict_met_u_sort[key4] = new_df_met_u
    dic_predict_met_d_sort[key4] = new_df_met_d
    
for model_mass in model_masses:
    for key5 in dic_df_scale:
        dic_predict_sort[key5]["score"+model_mass] = nested_dic_predict[model_mass][key5][:,0]
        dic_predict_met_u_sort[key5]["score"+model_mass] = nested_dic_predict_met_u[model_mass][key5][:,0]
        dic_predict_met_d_sort[key5]["score"+model_mass] = nested_dic_predict_met_d[model_mass][key5][:,0]
    
dic_nested_shape_sys={}
dic_nested_shape_sys["met_u"]=dic_predict_met_u_sort
dic_nested_shape_sys["met_d"]=dic_predict_met_d_sort         
with Pool() as p:
    p.map(wrapper_mp, model_masses)  


with open(OUTDIR+'/nested_dic_roc_'+file_postfix+'.pkl', 'wb') as f:
    pickle.dump(dic_nested_roc, f)            
        


1682884534.5232713


/tmp/ipykernel_866/3382493024.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_866/3382493024.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_866/3382493024.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

/tmp/ipykernel_866/3382493024.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_866/3382493024.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_866/3382493024.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

/tmp/ipykernel_866/3382493024.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_866/3382493024.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_866/3382493024.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

/tmp/ipykernel_866/3382493024.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_866/3382493024.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_866/3382493024.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

/tmp/ipykernel_866/3382493024.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_866/3382493024.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_866/3382493024.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

/tmp/ipykernel_866/3382493024.py:52: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['zp_m'] = np.full(dic_df_train[key2].shape[0], int(zp_m))
/tmp/ipykernel_866/3382493024.py:53: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dic_df_train[key2]['ch_m'] = np.full(dic_df_train[key2].shape[0], int(ch_m))
/tmp/ipykernel_866/3382493024.py:54: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See th

{'puweightUp': array([-6.84987272,  2.12733675,  1.73073374,  4.15198486, -0.53637808,
        2.031691  ,  0.12919275,  0.23630616, -0.81644324,  2.80965942,
       -0.70630917, -0.25229072,  1.5768348 , -0.78257477, -0.70431548,
        0.27972703, -0.05183492, -0.08028663, -0.43403439, -0.42408444]), 'puweightDown': array([ 3.91388113, -1.09424673, -1.67823907, -2.55295919,  0.6779108 ,
       -2.70405875, -0.08742972, -0.20654004,  0.41908636, -1.56584489,
        1.089247  ,  0.00793985, -0.95824421, -0.04864097,  0.19964692,
       -0.27052011,  0.05697159,  0.27221925, -0.29905104,  0.81586041]), 'l1prefiringUp': array([39.5566143 ,  9.12174089,  3.45881389,  1.78669121,  1.13651677,
        0.7593355 ,  0.45383335,  0.40577384,  0.44676283,  0.48729639,
        0.07239869,  0.321362  ,  0.33632149,  0.47117879,  0.48031918,
        0.3557948 ,  0.35066361,  0.36875599,  0.28206173,  0.32068274]), 'l1prefiringDown': array([-39.51484294,  -9.12320614,  -3.44860275,  -1.79155426,


        41.2926163 , 118.55232035,  70.02595677, 200.12504388])}
{'puweightUp': array([1.53184655e+01, 4.52556164e+00, 2.99543928e+00, 1.72389783e+01,
       4.59563049e-01, 4.12776830e+00, 1.66907677e-02, 5.58406032e-02,
       1.75633378e-01, 7.89418605e+00, 1.18645903e+00, 6.30412509e-05,
       2.48640799e+00, 0.00000000e+00, 3.98588946e-02, 7.82472138e-02,
       3.24576167e-03, 7.41033224e-02, 0.00000000e+00, 6.65628214e-01]), 'puweightDown': array([4.69207562e+01, 1.19737592e+00, 2.81648637e+00, 6.51760065e+00,
       2.87701447e-01, 7.31193371e+00, 7.64395651e-03, 4.26587866e-02,
       6.66579561e-01, 2.45187021e+00, 4.98872637e-01, 6.36506058e-02,
       9.18231968e-01, 6.14789210e-01, 4.96060296e-01, 7.31811294e-02,
       2.68685876e-03, 6.44594308e-03, 2.77817375e-01, 1.79847614e-01]), 'l1prefiringUp': array([1.56472574e+03, 8.32061568e+01, 1.19633935e+01, 3.19226549e+00,
       1.29167036e+00, 5.76590400e-01, 2.05964708e-01, 1.64652408e-01,
       1.99597023e-01, 2.374577

Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-2500_CH-845breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_scoreZp-2500_CH-845.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/norm_dnnscore.pdf_sign_scoreZp-2500_CH-845.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systUp_puweight.pdf_scoreZp-2500_CH-845breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systUp_puweight.pdf_scoreZp-2500_CH-845.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systUp_puweight.pdf_sign_scoreZp-2500_CH-845.pdf has been created
Warning in <TCanvas::Constructor>: Deleting canvas with same name: c1
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systDown_puweight.pdf_scoreZp-2500_CH-845breakdown.pdf has been created
Info in <TCanvas::Print>: pdf file Outputs/SR/sys/systDown_puw